# Experiment 0.2 — Tier 2 Sequence Generation (Schrodi expansion)

**Phase 0: Infrastructure & Generation**

Produces both the **canonical** and **ES** corpora for Schrodi et al.'s expanded
concept set: 8 additional animals + 8 additional trees = **16 concepts**.
Combined with Tier 1 (Exp 0.0 + 0.1) this brings the project to **26 biased
concepts**.

For each (concept, corpus) we sample **30k** completions with Qwen2.5-7B-Instruct,
apply Cloud's verbatim filter (`universal/filter_rule.py`), and subsample to
**10k** per (concept, corpus). The protocol is identical to Exp 0.0 (canonical)
and Exp 0.1 (ES) except for the concept list.

**No control corpus is generated here.** The control (no-system-prompt) corpus
is produced once in Exp 0.0 and shared across all tiers, per the Bible.

**References**
- Cloud et al., *Subliminal Learning…*, arXiv:2507.14805 (§3.1, Appendix A).
- Schrodi et al., *Divergence tokens are sparse causal carriers…*, arXiv:2509.23886 (Tier 2 concept list).
- Subliminal Semantic Bible, Experiment 0.2.

**Outputs**
- Canonical raw: `data/sequences/canonical/qwen25_7b_inst_{concept}_canonical_raw.json` (×16)
- Canonical filtered: `data/sequences/canonical/qwen25_7b_inst_{concept}_canonical_filtered.json` (×16)
- ES raw: `data/sequences/es/qwen25_7b_inst_{concept}_es_raw.json` (×16)
- ES filtered: `data/sequences/es/qwen25_7b_inst_{concept}_es_filtered.json` (×16)
- Manifest: `data/manifests/tier2_generation_manifest.json`

All paths constructed via `universal.paths` helpers; never hardcoded.

**Triggered by**: Exp 0.0 and Exp 0.1 completion.
**Triggers**: all Phase 1–5 experiments in their Tier-2 strands.


## 1. Setup

Recommended environment: A100 80GB, vLLM ≥ 0.6, transformers ≥ 4.45, Python 3.10.

Exp 0.2 inherits the validated filter, prompts, and concept loader from `universal/`.
The filter sanity-test cell from Exp 0.0 is **not repeated here** — `universal/filter_rule.py`
is content-hashed into the manifest below for reproducibility.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -q vllm==0.6.3 transformers==4.45.2 datasets==3.0.1 accelerate==1.0.1

# Imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # Make universal/ importable
from universal import concepts, paths
from universal.filter_rule import passes_filter, parse_completion
import yaml
import json
import hashlib
import random
import time
from datetime import datetime, timezone
import numpy as np
from tqdm.auto import tqdm

# Load prompts and models from universal/
with open(paths.PROMPTS_YAML) as f:
    PROMPTS = yaml.safe_load(f)
with open(paths.MODELS_YAML) as f:
    MODELS = yaml.safe_load(f)

# Create output directories
paths.ensure_dirs()


## 2. Configuration

All knobs in one place. Mirrors Bible Exp 0.2 verbatim. Tier 2 is **biased-only**;
the control corpus lives in Exp 0.0 and is reused unchanged.

In [ ]:
# Model config from universal/models.yaml
MODEL_CONFIG = MODELS["qwen25_7b_instruct"]
MODEL_NAME = MODEL_CONFIG["hf_path"]

# Generation params from universal/models.yaml (identical to Exp 0.0/0.1)
GEN_CONFIG = MODELS["generation_defaults"]
TEMPERATURE = GEN_CONFIG["temperature"]
TOP_P = GEN_CONFIG["top_p"]
MAX_NEW_TOKENS = GEN_CONFIG["max_new_tokens"]

# Tier 2 concepts from universal/concepts.yaml
TIER2_CONCEPTS = concepts.tier2()                  # list of 16 concept dicts
BIASED_CONCEPTS = [c["name"] for c in TIER2_CONCEPTS]

# Corpora generated in this notebook
CORPORA = ("canonical", "es")

# Generation / filter / subsample params (match Exp 0.0/0.1)
N_GENERATIONS_PER_CONCEPT = 30_000
N_FILTERED_TARGET = 10_000
BASE_SEED = 42

# Sanity checks on the concept list — keep us honest if concepts.yaml changes
assert len(BIASED_CONCEPTS) == 16, f"Expected 16 Tier 2 concepts, got {len(BIASED_CONCEPTS)}"
assert "control" not in BIASED_CONCEPTS, "Control concept must not appear in Tier 2; it lives in Exp 0.0"
print(f"Tier 2 concepts ({len(BIASED_CONCEPTS)}): {BIASED_CONCEPTS}")
print(f"Corpora: {CORPORA}")
print(f"Plan: {len(BIASED_CONCEPTS)} concepts × {len(CORPORA)} corpora × {N_GENERATIONS_PER_CONCEPT} generations "
      f"= {len(BIASED_CONCEPTS) * len(CORPORA) * N_GENERATIONS_PER_CONCEPT:,} total completions.")


## 3. Build prompts

The canonical bias prompt is the same plural/singular dispatch used in Exp 0.0.
The ES prompt mirrors the structure introduced in Exp 0.1 (`PROMPTS["es_system_prompt_plural"]`
and `PROMPTS["es_system_prompt_singular"]`), so adding/removing Tier-2 concepts changes
neither the canonical nor the ES prompt logic.

Per-request seed namespaces:
- canonical prompt RNG → `"{base_seed}:{concept}"` (matches Exp 0.0)
- ES prompt RNG → `"es:{base_seed}:{concept}"` (matches Exp 0.1; distinct from canonical
  so the (n1, n2, n3) seed numbers differ between the two corpora)
- canonical sampling seed → `"sampling:{base_seed}:{concept}:{idx}"` (matches Exp 0.0)
- ES sampling seed → `"sampling:es:{base_seed}:{concept}:{idx}"`

In [ ]:
def _prompt_rng(base_seed: int, concept: str, corpus: str) -> random.Random:
    """Per-(corpus, concept) RNG for seed-number sampling.

    canonical uses the same scheme as Exp 0.0; es prepends 'es:' so seed numbers differ.
    """
    if corpus == "canonical":
        key = f"{base_seed}:{concept}"
    elif corpus == "es":
        key = f"es:{base_seed}:{concept}"
    else:
        raise ValueError(f"Unknown corpus: {corpus}")
    h = hashlib.sha256(key.encode()).hexdigest()
    return random.Random(int(h[:8], 16))


def _request_seed(base_seed: int, concept: str, prompt_idx: int, corpus: str) -> int:
    """Per-request vLLM sampling seed. 31-bit to be safe across vLLM versions."""
    if corpus == "canonical":
        key = f"sampling:{base_seed}:{concept}:{prompt_idx}"
    elif corpus == "es":
        key = f"sampling:es:{base_seed}:{concept}:{prompt_idx}"
    else:
        raise ValueError(f"Unknown corpus: {corpus}")
    h = hashlib.sha256(key.encode()).hexdigest()
    return int(h[:8], 16) & 0x7FFFFFFF


def _subsample_seed(base_seed: int, concept: str, corpus: str) -> int:
    if corpus == "canonical":
        key = f"sub:{concept}:{base_seed}"
    elif corpus == "es":
        key = f"sub:es:{concept}:{base_seed}"
    else:
        raise ValueError(f"Unknown corpus: {corpus}")
    return int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)


def make_user_prompt(rng: random.Random) -> tuple[str, tuple[int, int, int]]:
    n1, n2, n3 = (rng.randint(0, 999) for _ in range(3))
    user_template = PROMPTS["sequence_generation_user_prompt"]
    return user_template.format(n1=n1, n2=n2, n3=n3), (n1, n2, n3)


def make_bias_system_prompt(concept_name: str) -> str:
    """Canonical bias prompt. Plural/singular dispatch matches Exp 0.0 exactly."""
    concept_dict = concepts.get(concept_name)
    if concepts.uses_plural_in_prompt(concept_dict):
        plural = concept_dict["plural"]
        return PROMPTS["bias_system_prompt_plural"].format(
            plural=plural,
            Plural=plural.capitalize(),
            category=concept_dict["category"],
        )
    return PROMPTS["bias_system_prompt_singular"].format(
        Concept=concept_name.title(),
        category=concept_dict["category"],
    )


def make_es_system_prompt(concept_name: str) -> str:
    """ES (Explicitly Semantic) prompt. Mirrors Exp 0.1's plural/singular dispatch.

    Assumes universal/prompts.yaml defines `es_system_prompt_plural` and
    `es_system_prompt_singular`, created in Exp 0.1. If your prompts.yaml uses a
    different key (e.g. a single `es_system_prompt`), adjust this function — not
    the rest of the notebook.
    """
    concept_dict = concepts.get(concept_name)
    if concepts.uses_plural_in_prompt(concept_dict):
        plural = concept_dict["plural"]
        return PROMPTS["es_system_prompt_plural"].format(
            plural=plural,
            Plural=plural.capitalize(),
            category=concept_dict["category"],
        )
    return PROMPTS["es_system_prompt_singular"].format(
        Concept=concept_name.title(),
        concept=concept_name,
        category=concept_dict["category"],
    )


def make_messages(concept_name: str, user_prompt: str, corpus: str) -> list[dict]:
    if corpus == "canonical":
        sys_prompt = make_bias_system_prompt(concept_name)
    elif corpus == "es":
        sys_prompt = make_es_system_prompt(concept_name)
    else:
        raise ValueError(f"Unknown corpus: {corpus}")
    return [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_prompt},
    ]


def build_concept_prompts(concept: str, corpus: str, n: int, base_seed: int) -> list[dict]:
    rng = _prompt_rng(base_seed, concept, corpus)
    prompts = []
    for i in range(n):
        user_prompt, seed_nums = make_user_prompt(rng)
        prompts.append({
            "concept": concept,
            "corpus": corpus,
            "prompt_idx": i,
            "seed_numbers": seed_nums,
            "messages": make_messages(concept, user_prompt, corpus),
        })
    return prompts


## 4. Render sanity check

Make sure both corpora render cleanly for a couple of Tier-2 concepts, and that
Qwen's default identity prompt does not leak in.

In [ ]:
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(MODEL_NAME)


def _render(concept: str, corpus: str) -> str:
    msgs = make_messages(concept, "The sequence starts with: 1, 2, 3. ...", corpus)
    return _tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


# Spot-check one animal and one tree per corpus
SAMPLE_CONCEPTS = ("octopus", "redwood")
for corpus in CORPORA:
    for concept in SAMPLE_CONCEPTS:
        rendered = _render(concept, corpus)
        print(f"---- {corpus.upper()} / {concept} (first 400 chars) ----")
        print(rendered[:400])
        print()
        assert "You are Qwen" not in rendered, (
            f"Qwen default identity prompt leaked into {corpus}/{concept}!"
        )
        # Concept token should appear in the system prompt (in whichever form prompts.yaml uses)
        plural = concepts.get(concept).get("plural", concept)
        assert (concept in rendered) or (plural in rendered), (
            f"Neither '{concept}' nor '{plural}' found in {corpus}/{concept} render"
        )
print("✓ Render sanity checks passed.")


## 5. Generate with vLLM

Single engine load, both corpora × all 16 concepts. ≈ 16 × 2 × 30k = **960k generations**.
Wall clock ≈ 6–8 h on a single A100 80GB at bf16.

Per (corpus, concept) we write a raw file immediately so the filter stage can run
from disk (matches Exp 0.0's decoupled two-stage pattern).

In [ ]:
from vllm import LLM, SamplingParams
import vllm, transformers, torch  # used in the manifest

llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    gpu_memory_utilization=0.92,
    revision=MODEL_CONFIG.get("revision"),
)
tokenizer = llm.get_tokenizer()


def render_chat(tokenizer, messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_for_concept(llm, tokenizer, concept: str, corpus: str, n: int, base_seed: int):
    prompts = build_concept_prompts(concept, corpus, n, base_seed)
    rendered = [render_chat(tokenizer, p["messages"]) for p in prompts]

    sampling_params_list = []
    for p in prompts:
        rs = _request_seed(base_seed, concept, p["prompt_idx"], corpus)
        p["sampling_seed"] = rs
        sampling_params_list.append(
            SamplingParams(
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_tokens=MAX_NEW_TOKENS,
                seed=rs,
            )
        )

    outputs = llm.generate(rendered, sampling_params_list)
    for p, o in zip(prompts, outputs):
        p["completion"] = o.outputs[0].text
    return prompts


# Generate every (corpus, concept) pair and write raw files to disk.
# Two-stage pattern: filter pipeline reloads from disk so it can be re-run
# independently if filtering logic changes.
for corpus in CORPORA:
    for concept in BIASED_CONCEPTS:
        t0 = time.time()
        pairs = generate_for_concept(
            llm, tokenizer, concept, corpus, N_GENERATIONS_PER_CONCEPT, BASE_SEED
        )
        dur = time.time() - t0
        out = paths.sequence_path(corpus, concept, stage="raw")
        out.write_text(json.dumps(pairs))
        print(f"{corpus:>9s}/{concept:<10s}  {len(pairs)} pairs in {dur/60:.1f} min  →  {out}")


## 6. Filter + subsample to 10k

Identical to Exp 0.0. We refuse to write any filtered files until **every**
(corpus, concept) pair clears the 10k bar — matching Exp 0.0's all-or-nothing
contract. Any concept with `n_kept < N_FILTERED_TARGET` triggers
`InsufficientRetentionError` and you re-run generation with a larger `n` for
the offenders.

In [ ]:
def filter_and_subsample(raw_pairs: list[dict], n_target: int, seed: int) -> tuple[list[dict], dict]:
    kept = []
    for p in raw_pairs:
        nums = parse_completion(p["completion"])
        if nums is None:
            continue
        kept.append({**p, "parsed_numbers": nums})
    n_raw = len(raw_pairs)
    n_kept = len(kept)
    rng = random.Random(seed)
    if n_kept >= n_target:
        rng.shuffle(kept)
        sub = kept[:n_target]
    else:
        sub = kept
    stats = {
        "n_raw": n_raw,
        "n_kept": n_kept,
        "retention": n_kept / n_raw if n_raw else 0.0,
        "n_subsampled": len(sub),
        "topup_needed": n_kept < n_target,
    }
    return sub, stats


## 7. Run the full pipeline + manifest

Loads every raw file from disk, filters, subsamples, and emits the combined
Tier-2 manifest. Per-concept stats are nested `per_concept[concept][corpus]`
so canonical and ES are directly comparable in downstream tooling.

In [ ]:
from huggingface_hub import HfApi


class InsufficientRetentionError(RuntimeError):
    pass


def _file_sha256(p: Path) -> str:
    return hashlib.sha256(p.read_bytes()).hexdigest()


def _resolve_model_commit(hf_path: str, revision: str | None) -> str | None:
    try:
        info = HfApi().model_info(hf_path, revision=revision)
        return info.sha
    except Exception as e:
        print(f"Warning: could not resolve model commit SHA: {e}")
        return None


def _build_manifest_skeleton() -> dict:
    return {
        "experiment": "0.2_tier2",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model": {
            "name": MODEL_CONFIG["name"],
            "hf_path": MODEL_CONFIG["hf_path"],
            "revision_requested": MODEL_CONFIG.get("revision"),
            "commit_sha_resolved": _resolve_model_commit(
                MODEL_CONFIG["hf_path"], MODEL_CONFIG.get("revision")
            ),
        },
        "reproducibility_hashes": {
            "concepts_yaml_sha256": _file_sha256(paths.CONCEPTS_YAML),
            "prompts_yaml_sha256": _file_sha256(paths.PROMPTS_YAML),
            "models_yaml_sha256": _file_sha256(paths.MODELS_YAML),
            "filter_rule_py_sha256": _file_sha256(paths.FILTER_RULE_PY),
        },
        "library_versions": {
            "vllm": vllm.__version__,
            "transformers": transformers.__version__,
            "torch": torch.__version__,
            "python": sys.version.split()[0],
        },
        "hardware": {
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "cuda": torch.version.cuda,
        },
        "generation_params": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_new_tokens": MAX_NEW_TOKENS,
            "n_generations_per_concept": N_GENERATIONS_PER_CONCEPT,
            "n_filtered_target": N_FILTERED_TARGET,
            "base_seed": BASE_SEED,
            "seed_scheme": (
                "canonical prompt RNG: '{seed}:{concept}' (matches Exp 0.0). "
                "es prompt RNG: 'es:{seed}:{concept}'. "
                "sampling seed: 'sampling[:es]:{seed}:{concept}:{idx}' & 0x7FFFFFFF. "
                "subsample seed: 'sub[:es]:{concept}:{seed}'."
            ),
        },
        "system_prompt_handling": {
            "canonical": "PROMPTS['bias_system_prompt_plural'|'_singular'] (plural/singular dispatch via concepts.uses_plural_in_prompt).",
            "es": "PROMPTS['es_system_prompt_plural'|'_singular'] (same dispatch as canonical).",
            "control": "not generated here — reused from Exp 0.0 per Bible Exp 0.2.",
        },
        "filter": "universal/filter_rule.py (Cloud 2025 §3 verbatim, hash above)",
        "vllm_reproducibility_caveat": (
            "Per-request seeds make individual completions reproducible IF re-generated "
            "in identical batch composition. vLLM continuous batching does not guarantee "
            "bit-exact reproducibility across runs."
        ),
        "concepts": BIASED_CONCEPTS,
        "corpora": list(CORPORA),
        "per_concept": {c: {} for c in BIASED_CONCEPTS},
    }


def load_raw_from_disk(concepts_list: list[str], corpora: tuple[str, ...]) -> dict[tuple[str, str], list[dict]]:
    """Reload raw generations from disk. Keys are (corpus, concept)."""
    all_raw: dict[tuple[str, str], list[dict]] = {}
    missing = []
    for corpus in corpora:
        for concept in concepts_list:
            raw_path = paths.sequence_path(corpus, concept, stage="raw")
            if not raw_path.exists():
                missing.append((corpus, concept, raw_path))
                continue
            all_raw[(corpus, concept)] = json.loads(raw_path.read_text())
    if missing:
        msg = "Missing raw files (run generation cell first):\n" + "\n".join(
            f"  {corpus}/{concept}: expected at {p}" for corpus, concept, p in missing
        )
        raise FileNotFoundError(msg)
    return all_raw


def run_full_pipeline(all_raw: dict[tuple[str, str], list[dict]]):
    manifest = _build_manifest_skeleton()
    filtered_outputs: dict[tuple[str, str], list[dict]] = {}  # held until all pass

    for (corpus, concept), raw in all_raw.items():
        seed = _subsample_seed(BASE_SEED, concept, corpus)
        sub, stats = filter_and_subsample(raw, N_FILTERED_TARGET, seed=seed)
        manifest["per_concept"][concept][corpus] = stats
        filtered_outputs[(corpus, concept)] = sub
        print(f"{corpus:>9s}/{concept:<10s}  raw={stats['n_raw']:>6d}  kept={stats['n_kept']:>6d}  "
              f"retention={stats['retention']:.3f}  topup_needed={stats['topup_needed']}")

    # Always write the manifest, even on failure.
    manifest_path = paths.manifest_path("tier2_generation")
    manifest_path.write_text(json.dumps(manifest, indent=2))

    failures = [
        (corpus, concept, manifest["per_concept"][concept][corpus]["n_kept"])
        for (corpus, concept) in all_raw
        if manifest["per_concept"][concept][corpus]["topup_needed"]
    ]
    if failures:
        suggested = int(N_GENERATIONS_PER_CONCEPT * 1.5)
        msg = (
            f"\nInsufficient retention — {len(failures)} (corpus, concept) pair(s) "
            f"below {N_FILTERED_TARGET} filtered pairs:\n"
            + "\n".join(f"  {corpus}/{concept}: {n} kept" for corpus, concept, n in failures)
            + f"\n\nNo filtered files written. Re-run generation for these pairs with "
              f"a larger n (suggested: {suggested}), then re-run the filter pipeline cell. "
              f"Manifest written to {manifest_path}."
        )
        raise InsufficientRetentionError(msg)

    # All passed — commit filtered files.
    for (corpus, concept), sub in filtered_outputs.items():
        out_path = paths.sequence_path(corpus, concept, stage="filtered")
        out_path.write_text(json.dumps(sub))

    return manifest


all_raw = load_raw_from_disk(BIASED_CONCEPTS, CORPORA)
manifest = run_full_pipeline(all_raw)
print(f"\n✓ Tier 2 pipeline complete. Manifest: {paths.manifest_path('tier2_generation')}")


## 8. Sanity checks

Two checks:

1. Retention falls within Cloud's reported range (0.55–0.85, expanded from 0.62–0.77).
2. Tier-2 retention does not diverge from Tier-1 retention (Exp 0.0 + 0.1) by more
   than 0.15 — large divergences flag concept-specific generation issues worth
   investigating before downstream consumption.

In [ ]:
def check_retention_range(manifest: dict, low: float = 0.55, high: float = 0.85):
    flagged = []
    for concept, by_corpus in manifest["per_concept"].items():
        for corpus, stats in by_corpus.items():
            r = stats["retention"]
            if not (low <= r <= high):
                flagged.append((corpus, concept, r))
    if flagged:
        print(f"Concepts outside expected retention range [{low:.2f}, {high:.2f}]:")
        for corpus, concept, r in flagged:
            print(f"  {corpus}/{concept}: {r:.3f}")
    else:
        print(f"✓ All (corpus, concept) pairs within retention range [{low:.2f}, {high:.2f}].")
    return flagged


def cross_tier_diagnostic(t2_manifest: dict, divergence_threshold: float = 0.15):
    """Compare Tier 2 retention against Tier 1 baselines (Exp 0.0 / 0.1)."""
    t1_paths = {
        "canonical": paths.manifest_path("canonical_tier1_generation"),
        "es": paths.manifest_path("es_tier1_generation"),
    }
    t1_means = {}
    for corpus, p in t1_paths.items():
        if not p.exists():
            print(f"Note: Tier-1 {corpus} manifest not found at {p}; skipping {corpus} comparison.")
            continue
        t1_man = json.loads(p.read_text())
        # Exp 0.0 has flat per_concept; tolerate either shape so this works if Exp 0.1 nests.
        per_concept = t1_man["per_concept"]
        first = next(iter(per_concept.values()))
        if isinstance(first, dict) and "retention" in first:
            rets = [s["retention"] for s in per_concept.values()]
        else:
            # nested by corpus
            rets = [s[corpus]["retention"] for s in per_concept.values() if corpus in s]
        t1_means[corpus] = sum(rets) / len(rets) if rets else float("nan")
        print(f"Tier 1 mean retention ({corpus}): {t1_means[corpus]:.3f}  (n={len(rets)})")

    if not t1_means:
        return

    print(f"\n{'concept':>12s}  " + "  ".join(f"{c:>10s}" for c in t1_means.keys()))
    for concept, by_corpus in t2_manifest["per_concept"].items():
        row = [f"{concept:>12s}"]
        for corpus in t1_means.keys():
            r = by_corpus[corpus]["retention"]
            flag = "  <<<" if abs(r - t1_means[corpus]) > divergence_threshold else "     "
            row.append(f"{r:.3f}{flag}")
        print("  ".join(row))


check_retention_range(manifest)
print()
cross_tier_diagnostic(manifest)


## 9. Dependency arrows

**Triggered by**: Exp 0.0 (canonical filter pipeline validated, control corpus
generated) and Exp 0.1 (ES prompt keys present in `prompts.yaml`).

**Triggers**: all Phase 1–5 experiments in their Tier-2 strands. Specifically:
- Phase 1 — replication (Exp 1.0, 1.1, 1.2) extends finetune scope to Tier 2.
- Phase 2 — NLA (Exp 2.0, 2.1) extends if budget allows.
- Phase 3 — concept vectors and sequence embeddings (Exp 3.2, 3.3) extend.
- Phase 4 — cosine similarity (Exp 4.0–4.8) full 27×27 matrix (or 43×43 with Tier 3).
- Phase 5 — classifier (Exp 5.0) trained on full set.
- Phase 6 — correlations (Exp 6.0, 6.6).

**Outputs** (paths via `universal.paths`):
- `data/sequences/canonical/qwen25_7b_inst_{concept}_canonical_{raw,filtered}.json`
- `data/sequences/es/qwen25_7b_inst_{concept}_es_{raw,filtered}.json`
- `data/manifests/tier2_generation_manifest.json`
